In [2]:
from diffusers import StableDiffusionPipeline
import torch

In [3]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    torch_dtype=torch.float16
).to("mps")


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [34]:
def poem_to_prompt(poem_text: str) -> str:
    # 也可以用 ChatGPT API 自动翻译，这里手动来
    translation = {
        "孤舟蓑笠翁，独钓寒江雪": "an old fisherman on a lonely boat, fishing alone in the snowy river",
        "千山鸟飞绝，万径人踪灭": "endless mountains with no birds in the sky, countless paths without a single soul",
        "白日依山尽，黄河入海流": "the sun sets behind the mountains, the Yellow River flows into the sea"
    }

    english_line = translation.get(poem_text, "a poetic Chinese ink painting of nature")
    return f"A painting, depicting: {english_line}"


In [1]:
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    torch_dtype=torch.float16
).to("mps")

def generate_poem_image(poem_text: str, output_path="poem_sd.png"):
    prompt = poem_to_prompt(poem_text)
    image = pipe(prompt).images[0]
    image.save(output_path)
    print(f"生成成功：{output_path}")


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [4]:
pipe.save_pretrained("./models/diffusions")

In [36]:
import requests

def translate_poem_with_deepseek(poem_text: str, api_key: str) -> str:
    url = "https://api.deepseek.com/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "deepseek-chat",  # 或 deepseek-coder，如果你想试试别的
        "messages": [
            {"role": "system", "content": "你是一个翻译家，请将一首古诗翻译成富有画面感和意境的英文描述，用于画图。注意描述词不能超过60个英文单词"},
            {"role": "user", "content": poem_text}
        ],
        "temperature": 0.7,
    }

    response = requests.post(url, headers=headers, json=data)
    
    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"].strip()
    else:
        raise RuntimeError(f"请求失败，状态码 {response.status_code}，响应内容：{response.text}")


In [37]:
api_key = "sk-164ad8ec739c466aa7a53489f3f9eaaa"
poem = "借问酒家何处有，牧童遥指杏花村"

english = translate_poem_with_deepseek(poem, api_key)
print("翻译结果：", english)

翻译结果： A young shepherd boy points into the distance, where a village of blooming apricot trees emerges through spring mist. His gesture invites travelers to seek warmth and wine beyond the winding path.


In [38]:
def poem_to_prompt(poem_text: str) -> str:
    english = translate_poem_with_deepseek(poem_text, api_key)
    return f"A painting, depicting: {english}"

def generate_poem_image(poem_text: str, output_path="poem_sd.png"):
    prompt = poem_to_prompt(poem_text)
    print(f"生成 prompt: {prompt}")
    image = pipe(prompt).images[0]
    image.save(output_path)
    print(f"图像已保存至：{output_path}")



In [39]:
generate_poem_image("借问酒家何处有，牧童遥指杏花村", "output_wine2.png")

生成 prompt: A painting, depicting: A traveler asks a shepherd boy for directions to a tavern. The child points far into the distance, where a village emerges amidst blooming apricot trees, their pink blossoms glowing in the soft evening light.


  0%|          | 0/50 [00:00<?, ?it/s]

图像已保存至：output_wine2.png
